In [ ]:
import requests
import time

#  Step 1: Add your Azure details
endpoint = "https://shabbubab123.cognitiveservices.azure.com/"
key = "key"

url = f"{endpoint}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

headers = {
    "Content-Type": "application/pdf",
    "Ocp-Apim-Subscription-Key": key
}

file_path = "Insurance claim form.pdf"

#  Step 2: Send request
with open(file_path, "rb") as f:
    response = requests.post(url, headers=headers, data=f)

print("Status:", response.status_code)
print("Response:", response.text)

#  Stop if request failed
if response.status_code != 202:
    print(" Request failed. Check file, key, or endpoint.")
    exit()

#  Step 3: Get operation URL safely
operation_url = response.headers.get("operation-location")

if not operation_url:
    print(" operation-location not found")
    exit()

#  Step 4: Poll until completed
while True:
    result = requests.get(operation_url, headers={"Ocp-Apim-Subscription-Key": key})
    result_json = result.json()

    if result_json["status"] == "succeeded":
        break
    elif result_json["status"] == "failed":
        print(" Analysis failed")
        exit()

    time.sleep(2)

#  Step 5: Extract full text (for debugging)
content = result_json["analyzeResult"]["content"]



#  Step 6: Try structured extraction (BEST METHOD)
import re

text = content

#  NAME → pick Rahul Kumar (not S. Sharma)
name = ""

lines = content.split("\n")

for i, line in enumerate(lines):
    if "CLAIMANT NAME" in line.upper():
        # next line is actual name
        if i + 1 < len(lines):
            name = lines[i + 1].strip()
            break


#  POLICY NUMBER
policy_match = re.search(r"POL\d+", text)
policy_number = policy_match.group() if policy_match else ""


#  CLAIM AMOUNT (₹ 75,000)
amount_match = re.search(r"CLAIM AMOUNT.*?₹\s?[\d,]+", text, re.DOTALL)
claim_amount = ""

if amount_match:
    claim_amount = re.search(r"₹\s?[\d,]+", amount_match.group()).group()


#  DATE → pick last valid date (not DOB, not loss date)
dates = re.findall(r"\d{2}/\d{2}/\d{4}", text)
date = dates[-1] if dates else ""


# FINAL OUTPUT
print("\nExtracted Information:")
print("Name:", name)
print("Claim Amount:", claim_amount)
print("Date:", date)
print("Policy Number:", policy_number)



Status: 202
Response: 

Extracted Information:
Name: RAHUL KUMAR
Claim Amount: ₹ 75,000
Date: 18/02/2025
Policy Number: POL123456789
